In [4]:
import os
import time
import json
import numpy as np
from pathlib import Path
from openai import OpenAI
from neo4j import GraphDatabase
from sentence_transformers import SentenceTransformer
import torch

PROJECT_ROOT = Path.home() / "xai-knowledge-graph"

def load_env(path):
    out = {}
    with open(path) as f:
        for line in f:
            line = line.strip()
            if "=" in line and not line.startswith("#"):
                k, v = line.split("=", 1)
                out[k.strip()] = v.strip()
    return out

env  = load_env(Path.home() / ".env_xai_kg")
aura = load_env(Path.home() / "aura_cred.txt")

llm = OpenAI(api_key=env["GROQ_API_KEY"], base_url="https://api.groq.com/openai/v1")
LLM_MODEL = "llama-3.3-70b-versatile"

driver = GraphDatabase.driver(aura["NEO4J_URI"], auth=(aura["NEO4J_USERNAME"], aura["NEO4J_PASSWORD"]))
DB = aura["NEO4J_DATABASE"]

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

# Same llm_call from Day 6 — copy it in
def llm_call(prompt: str, max_retries: int = 4) -> str:
    for attempt in range(max_retries):
        try:
            resp = llm.chat.completions.create(
                model=LLM_MODEL,
                messages=[{"role": "user", "content": prompt}],
                temperature=0.0,
                max_tokens=600,
            )
            time.sleep(2)
            return resp.choices[0].message.content.strip()
        except Exception as e:
            if "429" in str(e) or "rate_limit" in str(e).lower():
                wait = 20 * (attempt + 1)
                print(f"  ⚠ Rate limited — waiting {wait}s")
                time.sleep(wait)
            else:
                raise
    raise RuntimeError(f"LLM failed after {max_retries} retries")

Device: cuda


In [14]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.home() / "xai-knowledge-graph"
sys.path.insert(0, str(PROJECT_ROOT))

# Sanity checks
print("PROJECT_ROOT exists:", PROJECT_ROOT.exists())
print("src/ exists:", (PROJECT_ROOT / "src").exists())
print("src/graphrag.py exists:", (PROJECT_ROOT / "src" / "graphrag.py").exists())
print("src/__init__.py exists:", (PROJECT_ROOT / "src" / "__init__.py").exists())

from src.graphrag import setup, graphrag_answer, nl_to_cypher, validate_cypher

# Now setup() is defined and you can call it:
setup(
    llm_client=llm,
    neo4j_driver=driver,
    db_name=DB,
    llm_model=LLM_MODEL,
)

print("✓ graphrag module loaded and initialized")

PROJECT_ROOT exists: True
src/ exists: True
src/graphrag.py exists: True
src/__init__.py exists: True
✓ graphrag module loaded and initialized


In [7]:
print("Fetching papers + abstracts from Neo4j...")
papers = []
with driver.session(database=DB) as s:
    for r in s.run("MATCH (p:Paper) RETURN p.arxiv_id AS aid, p.title AS title, p.abstract AS abstract, p.year AS year, p.citation_count AS citations"):
        if r["abstract"] and len(r["abstract"]) > 50:  # filter out empties
            papers.append({
                "arxiv_id":  r["aid"],
                "title":     r["title"],
                "abstract":  r["abstract"],
                "year":      r["year"],
                "citations": r["citations"] or 0,
            })

print(f"Loaded {len(papers):,} papers with abstracts")

Fetching papers + abstracts from Neo4j...
Loaded 3,907 papers with abstracts


In [8]:
EMB_MODEL = "intfloat/multilingual-e5-base"  
encoder = SentenceTransformer(EMB_MODEL, device=DEVICE)

# E5 models expect "passage: " prefix for documents, "query: " for queries
passages = [f"passage: {p['title']}. {p['abstract']}" for p in papers]

print(f"Encoding {len(passages):,} passages...")
embeddings = encoder.encode(
    passages, 
    batch_size=64, 
    show_progress_bar=True, 
    normalize_embeddings=True,   # normalize so dot product = cosine
    convert_to_numpy=True,
)
print(f"Embeddings shape: {embeddings.shape}")

# Save for reuse
EMB_PATH = PROJECT_ROOT / "data" / "rag_embeddings.npz"
EMB_PATH.parent.mkdir(parents=True, exist_ok=True)
np.savez_compressed(EMB_PATH, embeddings=embeddings)
print(f"Saved → {EMB_PATH}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Encoding 3,907 passages...


Batches:   0%|          | 0/62 [00:00<?, ?it/s]

Embeddings shape: (3907, 768)
Saved → /slurm/homes/pchandra/xai-knowledge-graph/data/rag_embeddings.npz


In [9]:
def retrieve(question: str, k: int = 5) -> list[dict]:
    """Retrieve top-k papers by cosine similarity to the question."""
    q_emb = encoder.encode(
        f"query: {question}", 
        normalize_embeddings=True, 
        convert_to_numpy=True,
    )
    sims = embeddings @ q_emb          # dot product = cosine (both normalized)
    top_indices = sims.argsort()[::-1][:k]
    return [{**papers[i], "similarity": float(sims[i])} for i in top_indices]

In [10]:
RAG_ANSWER_PROMPT = """You are answering a question about Explainable AI research.

IMPORTANT: This corpus contains exclusively Explainable AI (XAI) research papers from arXiv.
All papers, authors, and topics are XAI-related by construction.

User question: {question}

The following {k} papers were retrieved by semantic similarity to the question:

{context}

Based ONLY on the abstracts and titles above, provide a clear, concise answer.
If the retrieved papers don't fully answer the question, say so honestly.
Do not invent facts not present in the retrieved context.
"""

def format_rag_context(retrieved: list[dict]) -> str:
    lines = []
    for i, p in enumerate(retrieved, 1):
        lines.append(
            f"[{i}] ({p['year']}, {p['citations']} citations, sim={p['similarity']:.3f})\n"
            f"    Title:    {p['title']}\n"
            f"    Abstract: {p['abstract'][:500]}{'...' if len(p['abstract']) > 500 else ''}\n"
        )
    return "\n".join(lines)

def vanilla_rag_answer(question: str, k: int = 5) -> dict:
    """End-to-end vanilla RAG: question → semantic retrieval → LLM answer."""
    retrieved = retrieve(question, k=k)
    context   = format_rag_context(retrieved)
    
    prompt = RAG_ANSWER_PROMPT.format(question=question, k=k, context=context)
    answer = llm_call(prompt)
    
    return {
        "question":     question,
        "retrieved":    [{"title": p["title"], "similarity": p["similarity"]} for p in retrieved],
        "context_preview": context[:600] + ("..." if len(context) > 600 else ""),
        "answer":       answer,
    }

# Smoke test
result = vanilla_rag_answer("What is SHAP and how does it work?")
print("Q:", result["question"])
print("\nRetrieved:")
for r in result["retrieved"]:
    print(f"  [{r['similarity']:.3f}]  {r['title'][:80]}")
print(f"\nAnswer:\n{result['answer']}")

Q: What is SHAP and how does it work?

Retrieved:
  [0.871]  SHAP values via sparse Fourier representation
  [0.864]  Verified SHAP: Provable Bounds for Exact Shapley Values of Neural Networks
  [0.862]  Causal SHAP: Feature Attribution with Dependency Awareness through Causal Discov
  [0.859]  Towards Piece-by-Piece Explanations for Chess Positions with SHAP
  [0.858]  Fool SHAP with Stealthily Biased Sampling

Answer:
SHAP (SHapley Additive exPlanations) is a method for local feature attribution in interpretable and explainable AI. It is used to explain machine learning predictions by attributing the contribution of each feature to the predicted outcome. The retrieved papers provide various approaches to computing or utilizing SHAP values, such as using sparse Fourier representation, verifying provable bounds for exact SHAP values, integrating causal relationships into feature attribution, and applying SHAP to specific domains like chess analysis. However, the papers do not provide a

In [11]:
TEST_QUESTIONS = [
    # 1-3: factual / aggregation — GraphRAG should dominate
    "How many papers in this corpus discuss SHAP?",
    "List the top 5 most cited papers in this corpus.",
    "Who has written the most papers about Counterfactual explanations?",
    
    # 4-5: multi-hop — GraphRAG should dominate
    "Find papers that cite Grad-CAM and are about Healthcare applications.",
    "Which authors collaborate most often in this corpus?",
    
    # 6-7: conceptual / synthesis — vanilla RAG should do better
    "Based on these papers, what is the difference between LIME and SHAP?",
    "What are the key challenges in evaluating XAI methods according to this research?",
    
    # 8: recent trends — could go either way
    "What recent advances have been made in counterfactual explanations?",
    
    # 9-10: harder, mixed reasoning
    "How does this corpus describe the relationship between Interpretability and Computer Vision research?",
    "What recent papers focus on transformer-based XAI methods?",
]

In [15]:
# Import the GraphRAG function — copy graphrag_answer + dependencies into this notebook
# Or, cleaner: import from your saved src/graphrag.py module

# Re-define here for clarity:
#SYSTEM_PROMPT = """..."""   # paste from GraphRag notebook
#ANSWER_PROMPT = """..."""   # paste from GraphRag notebook 
#FORBIDDEN_KEYWORDS = {...}  # paste from GraphRag notebook

# Then re-define: nl_to_cypher, execute_cypher, validate_cypher, 
# format_results_as_context, graphrag_answer (paste from GraphRag notebook)

# === Run both pipelines on all questions ===
results = []
for i, q in enumerate(TEST_QUESTIONS, 1):
    print(f"\n{'='*100}")
    print(f"Q{i}: {q}")
    print(f"{'='*100}")
    
    print("\n--- GraphRAG ---")
    gr = graphrag_answer(q)
    print(f"Cypher:\n{gr['cypher']}\n")
    print(f"Answer:\n{gr.get('answer', '(error)')}\n")
    
    print("--- Vanilla RAG ---")
    vr = vanilla_rag_answer(q)
    print(f"Top retrieved:")
    for r in vr["retrieved"]:
        print(f"  [{r['similarity']:.3f}] {r['title'][:80]}")
    print(f"\nAnswer:\n{vr['answer']}")
    
    results.append({"q_num": i, "question": q, "graphrag": gr, "vanilla_rag": vr})

# Save
RESULTS_PATH = PROJECT_ROOT / "data" / "rag_comparison_results.json"
with open(RESULTS_PATH, "w") as f:
    json.dump(results, f, indent=2, default=str)
print(f"\n Saved {len(results)} comparison results to {RESULTS_PATH}")


Q1: How many papers in this corpus discuss SHAP?

--- GraphRAG ---
Cypher:
MATCH (p:Paper)-[:ABOUT]->(:Topic {name: "SHAP"})
RETURN count(DISTINCT p) AS papers LIMIT 1

Answer:
There are 1057 papers in this corpus that discuss SHAP.

--- Vanilla RAG ---
Top retrieved:
  [0.853] SHAP scores fail pervasively even when Lipschitz succeeds
  [0.850] Causal SHAP: Feature Attribution with Dependency Awareness through Causal Discov
  [0.847] SHAP values via sparse Fourier representation
  [0.843] Consistency of Feature Attribution in Deep Learning Architectures for Multi-Omic
  [0.842] Explaining Human Activity Recognition with SHAP: Validating Insights with Pertur

Answer:
All 5 papers in the retrieved set discuss SHAP (SHapley Additive exPlanations), as it is explicitly mentioned in their titles or abstracts. Therefore, the answer to the question is that at least 5 papers in this corpus discuss SHAP.

Q2: List the top 5 most cited papers in this corpus.

--- GraphRAG ---
Cypher:
MATCH (p:Pa

In [16]:
JUDGE_PROMPT = """You are evaluating two answers to the same question about Explainable AI research.

Question: {question}

Answer A (GraphRAG — uses structured graph queries):
{answer_a}

Answer B (Vanilla RAG — uses semantic retrieval over abstracts):
{answer_b}

Score each answer on a scale of 1–5 for each criterion. Be objective and concise.

Criteria (1=poor, 5=excellent):
- ACCURACY: Are the factual claims correct? Penalize hallucination or unsupported claims.
- COMPLETENESS: Does the answer fully address the question?
- SPECIFICITY: Does the answer use concrete facts (numbers, names, titles) vs vague generalities?
- GROUNDING: Are claims traceable to information that would be in the underlying data?

Respond in EXACTLY this format (no extra text):

A_accuracy: <1-5>
A_completeness: <1-5>
A_specificity: <1-5>
A_grounding: <1-5>
B_accuracy: <1-5>
B_completeness: <1-5>
B_specificity: <1-5>
B_grounding: <1-5>
winner: <A or B or tie>
reasoning: <one sentence>
"""

import re

def parse_judge(text: str) -> dict:
    """Parse the judge's structured response."""
    out = {}
    for line in text.strip().split("\n"):
        if ":" in line:
            k, v = line.split(":", 1)
            k = k.strip()
            v = v.strip()
            if k in ("winner", "reasoning"):
                out[k] = v
            else:
                m = re.search(r"\d+", v)
                if m:
                    out[k] = int(m.group())
    return out

# Score every question
scores = []
for r in results:
    print(f"Judging Q{r['q_num']}...")
    a_ans = r["graphrag"].get("answer", "(error)")
    b_ans = r["vanilla_rag"]["answer"]
    
    judge_text = llm_call(JUDGE_PROMPT.format(
        question=r["question"], answer_a=a_ans, answer_b=b_ans
    ))
    parsed = parse_judge(judge_text)
    parsed["q_num"]    = r["q_num"]
    parsed["question"] = r["question"]
    scores.append(parsed)

# Save scoring
with open(PROJECT_ROOT / "data" / "rag_comparison_scores.json", "w") as f:
    json.dump(scores, f, indent=2)

Judging Q1...
Judging Q2...
Judging Q3...
Judging Q4...
Judging Q5...
Judging Q6...
Judging Q7...
Judging Q8...
Judging Q9...
Judging Q10...


In [17]:
import pandas as pd

rows = []
for s in scores:
    rows.append({
        "Q": s["q_num"],
        "Question":            s["question"][:60] + ("..." if len(s["question"]) > 60 else ""),
        "GR_acc":  s.get("A_accuracy", 0),
        "GR_comp": s.get("A_completeness", 0),
        "GR_spec": s.get("A_specificity", 0),
        "GR_grnd": s.get("A_grounding", 0),
        "RAG_acc":  s.get("B_accuracy", 0),
        "RAG_comp": s.get("B_completeness", 0),
        "RAG_spec": s.get("B_specificity", 0),
        "RAG_grnd": s.get("B_grounding", 0),
        "Winner":   s.get("winner", "?"),
    })

df = pd.DataFrame(rows)
print(df.to_string(index=False))

# Headline numbers
print("\n=== AVERAGES ===")
print(f"GraphRAG  — accuracy: {df['GR_acc'].mean():.2f}, completeness: {df['GR_comp'].mean():.2f}, specificity: {df['GR_spec'].mean():.2f}, grounding: {df['GR_grnd'].mean():.2f}")
print(f"VanillaRAG — accuracy: {df['RAG_acc'].mean():.2f}, completeness: {df['RAG_comp'].mean():.2f}, specificity: {df['RAG_spec'].mean():.2f}, grounding: {df['RAG_grnd'].mean():.2f}")

print(f"\nWin counts:")
print(df['Winner'].value_counts().to_string())

# Save table
df.to_csv(PROJECT_ROOT / "data" / "rag_comparison_table.csv", index=False)
print(f"\nSaved comparison table.")


 Q                                                        Question  GR_acc  GR_comp  GR_spec  GR_grnd  RAG_acc  RAG_comp  RAG_spec  RAG_grnd Winner
 1                    How many papers in this corpus discuss SHAP?       1        5        5        1        5         3         5         5      B
 2                List the top 5 most cited papers in this corpus.       5        5        5        5        1         1         1         1      A
 3 Who has written the most papers about Counterfactual explana...       1        5        5        1        5         1         1         5      B
 4 Find papers that cite Grad-CAM and are about Healthcare appl...       4        5        5        4        5         2         4         5      A
 5            Which authors collaborate most often in this corpus?       5        5        5        5        5         1         1         1      A
 6 Based on these papers, what is the difference between LIME a...       1        1        1        1        4  